## **자족용지 비율 추가 (add_자족용지비율)**

#### **0. 라이브러리 임포트**

In [1]:
import pandas as pd

print("라이브러리 임포트 완료")

라이브러리 임포트 완료


#### **1. 설정**

- **파일 경로** 및 **도시명 → 사업명 매핑** 딕셔너리 정의
- `CITY_LUSE_MAP`: 도시명을 키(key), 해당 도시의 택지 사업명 튜플을 값(value)으로 구성

In [2]:
LUSE_CSV   = r'C:\3_1_DataMining\팀 프로젝트\토지이용계획\BLS5_DSTRC_LAND.csv'
INPUT_CSV  = r'C:\3_1_DataMining\팀 프로젝트\geomdan_merged.csv'   # 모든 도시 합쳐진 파일
OUTPUT_CSV = r'C:\3_1_DataMining\팀 프로젝트\geomdan_merged2.csv'

# 도시명 → 사업명 매핑
CITY_LUSE_MAP = {
    '청라': (
        '인천경제자유구역 청라국제도시',
    ),
    '송도': (
        '인천경제자유구역 송도국제도시 송도국제화복합단지',
        '인천경제자유구역 송도국제도시 송도랜드마크시티(6·8공구)',
        '인천경제자유구역 송도국제도시 첨단산업클러스터C ',
        '인천경제자유구역 송도국제도시 국제업무단지',
        '인천경제자유구역 송도국제도시 첨단산업클러스터(B)',
        '인천경제자유구역 송도국제도시 시가지조성단지',
        '인천경제자유구역 송도국제도시 지식정보산업단지',
    ),
    '판교': (
        '성남판교지구 택지개발사업',
    ),
    '운정': (
        '파주운정지구 택지개발사업',
        '파주운정3 택지개발사업',
    ),
    '광교': (
        '광교지구 택지개발사업',
    ),
    '계양': (
        '인천계양 테크노밸리 공공주택지구',
    ),
    '검단': (
        '인천검단지구 택지개발지구',
    ),
}

print(f"매핑 도시 수: {len(CITY_LUSE_MAP)}개")
print(f"대상 도시: {list(CITY_LUSE_MAP.keys())}")

매핑 도시 수: 7개
대상 도시: ['청라', '송도', '판교', '운정', '광교', '계양', '검단']


#### **2. 자족용지 비율 계산 함수 정의**

In [3]:
def calc_self_ratio(df_luse, luse_names):
    df_city = df_luse[df_luse['지구명'].isin(luse_names)]

    total_m2 = df_city['면적소계'].sum()
    self_m2  = df_city[
        (df_city['대분류'] == '산업시설') |
        (df_city['중분류'] == '업무시설용지')
    ]['면적소계'].sum()
    ratio    = self_m2 / total_m2 if total_m2 > 0 else 0

    return total_m2, self_m2, ratio

print("함수 정의 완료: calc_self_ratio")

함수 정의 완료: calc_self_ratio


#### **3. 택지정보 CSV 읽기**

In [4]:
print("택지정보 CSV 읽는 중...")
df_luse = pd.read_csv(LUSE_CSV, encoding='cp949')
print(f"읽기 완료: {len(df_luse):,}행")

택지정보 CSV 읽는 중...
읽기 완료: 17,135행


#### **4. 지구명 확인 (검단)**

- `str.contains()`: 지구명 컬럼에서 '검단' 문자열이 포함된 행을 필터링하여 실제 지구명 확인
- `unique()`: 중복 없이 지구명 목록 출력

In [5]:
# '검단' 포함된 지구명 전체 출력
candidates = df_luse[df_luse['지구명'].str.contains('검단', na=False)]['지구명'].unique()
for name in candidates:
    print(name)

인천 검단일반산업단지
인천검단지구 택지개발지구


#### **5. 도시별 자족용지 비율 계산**

In [6]:
print("=" * 60)
print("자족용지 비율 계산 중...")
print("=" * 60)

ratio_cache = {}
for city, luse_names in CITY_LUSE_MAP.items():
    total_m2, self_m2, ratio = calc_self_ratio(df_luse, luse_names)
    if ratio is not None:
        ratio_cache[city] = ratio
        print(f"[{city}]")
        print(f"  전체면적:     {total_m2:>15,.1f} ㎡  ({total_m2/1e6:.4f} km²)")
        print(f"  자족용지:     {self_m2:>15,.1f} ㎡  ({self_m2/1e6:.4f} km²)")
        print(f"  자족용지비율: {ratio:.6f}  ({ratio*100:.4f}%)")
        print()

자족용지 비율 계산 중...
[청라]
  전체면적:        17,804,756.8 ㎡  (17.8048 km²)
  자족용지:         1,854,933.2 ㎡  (1.8549 km²)
  자족용지비율: 0.104182  (10.4182%)

[송도]
  전체면적:        32,722,808.7 ㎡  (32.7228 km²)
  자족용지:         3,930,678.0 ㎡  (3.9307 km²)
  자족용지비율: 0.120120  (12.0120%)

[판교]
  전체면적:         8,921,788.2 ㎡  (8.9218 km²)
  자족용지:           488,342.6 ㎡  (0.4883 km²)
  자족용지비율: 0.054736  (5.4736%)

[운정]
  전체면적:        16,610,316.0 ㎡  (16.6103 km²)
  자족용지:           313,200.2 ㎡  (0.3132 km²)
  자족용지비율: 0.018856  (1.8856%)

[광교]
  전체면적:        10,786,948.6 ㎡  (10.7869 km²)
  자족용지:           669,751.0 ㎡  (0.6698 km²)
  자족용지비율: 0.062089  (6.2089%)

[계양]
  전체면적:         3,367,311.0 ㎡  (3.3673 km²)
  자족용지:           583,803.0 ㎡  (0.5838 km²)
  자족용지비율: 0.173374  (17.3374%)

[검단]
  전체면적:        11,109,137.3 ㎡  (11.1091 km²)
  자족용지:           764,385.5 ㎡  (0.7644 km²)
  자족용지비율: 0.068807  (6.8807%)



#### **6. 원본 CSV 읽기 및 비율 매핑**

- **6-1. 원본 CSV 읽기**

In [7]:
print("=" * 60)
print("원본 CSV 처리 중...")
print("=" * 60)

df = pd.read_csv(INPUT_CSV, encoding='utf-8')
print(f"읽기 완료: {len(df):,}행")

원본 CSV 처리 중...
읽기 완료: 35,614행


- **6-2. 도시명 기준 자족용지비율 컬럼 추가**

In [8]:
# 도시명 컬럼 기준으로 비율 매핑
df['자족용지비율'] = df['도시명'].map(ratio_cache)

# 매핑 안 된 도시 확인
unmatched = df[df['자족용지비율'].isna()]['도시명'].unique()
if len(unmatched) > 0:
    print(f"매핑 실패한 도시명: {unmatched}")
    print("CITY_LUSE_MAP의 키값과 CSV의 도시명이 일치하는지 확인하세요.")
else:
    print("모든 도시 매핑 완료")

모든 도시 매핑 완료


#### **7. 결과 저장**

In [9]:
df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')

print("=" * 60)
print(f"저장 완료 → {OUTPUT_CSV}")
print(f"전체 행 수: {len(df):,}행")

저장 완료 → C:\3_1_DataMining\팀 프로젝트\geomdan_merged2.csv
전체 행 수: 35,614행
